In [1]:
import json
import os
import pandas as pd
import numpy as np

In [2]:
emlap_metadata = pd.read_csv("/srv/data/tome/tome-corpus/emlap_metadata.csv", index_col=0, sep=";")
ids = emlap_metadata["no."]

In [3]:
emlap_metadata.head(10)

,working_title,filenames,no.,is_done,is_noscemus,if_noscemus_id,AUTHORSHIP,is_one_author,#if more than 1 author skip section and choose compendium below,is_author_known,...,CONTENTS,genre,subject,SOURCE OF FILE,link,source_of_file,origin_of_copy,other_notes,tokens_N,aurhor_wd
0,"Augurello, Chrysopoeia",100001_Augurello1515_Chrysopoeia_GB_Noscemus,100001,True,True,713324.0,NaN,True,NaN,True,...,NaN,didactic poem,alchemy,NaN,https://wiki.uibk.ac.at/noscemus/Chrysopoeia,GB,Noscemus,NaN,23718,NaN
1,"Pseudo-Lull, Secretis",100002_Pseudo-Lull1518_De secretis_naturae_MDZ...,100002,True,False,NaN,NaN,True,NaN,True,...,NaN,treatise,"alchemy, medicine",NaN,https://www.digitale-sammlungen.de/en/view/bsb...,MDZ,MBS,NaN,24673,NaN
2,"Pantheus, Ars Transmutatione",100003_Pantheus1518_Ars_Transmutationis_Metall...,100003,True,False,NaN,NaN,True,NaN,True,...,NaN,treatise,alchemy,NaN,https://www.google.co.uk/books/edition/Ars_Tra...,GB,BL,NaN,8646,NaN
3,"Anon, Vera alchemiae",100004_Anon1561_Verae_Alchemiae_MDZ_MBS,100004,True,False,NaN,NaN,True,NaN,True,...,NaN,"compendium, florilegium",alchemy,NaN,https://mdz-nbn-resolving.de/details:bsb10141168,MDZ,MBS,NaN,3521,NaN
4,"Pantheus, Voarchadumia",100005_Pantheus1530_Voarchadumia_ONB,100005,True,False,NaN,NaN,True,NaN,True,...,NaN,treatise,alchemy,NaN,https://data.onb.ac.at/rep/10588E49,ONB,ONB,NaN,20386,NaN
5,"Savonarola, De arte conficiendi aquam vitae",100006_Savonarola1532_De_arte_conficiendi_aqua...,100006,True,False,NaN,NaN,True,NaN,True,...,NaN,treatise,"alchemy, medicine",NaN,https://onb.digital/result/1088C33C,ONB,ONB,NaN,15237,NaN
6,"Anon, Rosarium philosophorum",100007_Anon1550_Rosarium_philosophorum_ER_ZZ,100007,True,False,NaN,NaN,False,NaN,False,...,NaN,"treatise, florilegium",alchemy,NaN,https://doi.org/10.3931/e-rara-34454,ER,ZZ,NaN,36755,NaN
7,"Severinus, Epistola",100008_Severinus1572_Epistola_MBZ_MBS,100008,True,False,NaN,NaN,True,NaN,True,...,NaN,letter,"alchemy, medicine, Paracelsianism",NaN,https://mdz-nbn-resolving.de/details:bsb00034170,MDZ,MBS,NaN,2265,NaN
8,"Vegius, Inter inferiora corpora",100009_Vegius1518_Inter_inferiora_corpora_disp...,100009,True,False,NaN,NaN,False,NaN,False,...,NaN,"treatise, dialogue, play",Renaissance philosophy,NaN,http://data.onb.ac.at/rep/1099703B,ONB,ONB,NaN,27934,NaN
9,"Bracesco, De alchimia dialogi duo",100010_Bracesco1548_De_alchemia_dialogi_duo_IA...,100010,True,False,NaN,NaN,True,NaN,True,...,NaN,dialogue,alchemy,NaN,https://archive.org/details/dealchemiadialo00b...,IA,UCM,NaN,42220,NaN


In [4]:
#
emlap_tokens_df = pd.read_parquet("../data/emlap_corpus_public/emlap_tokens_df.parquet")
emlap_tokens_df.head(5)

,token_text,lemma,pos,ref,char_start,char_end,id,sent_idx,sent_len,page,textblock,tag,blocktype
0,Germani,Germani,PROPN,"{'blocktype': 'text', 'page': [3], 'tag': '', ...",0,7,100044,0,7,[3],[1],,text
1,Paracelsi,Paracelsi,NOUN,"{'blocktype': 'title', 'page': [3], 'tag': '',...",0,9,100044,1,23,[3],[2],,title
2,",",",",PUNCT,"{'blocktype': 'title', 'page': [3], 'tag': '',...",9,10,100044,1,23,[3],[2],,title
3,Medicorum,medicus,NOUN,"{'blocktype': 'title', 'page': [3], 'tag': '',...",11,20,100044,1,23,[3],"[2, 3]",,title
4,Et,et,CCONJ,"{'blocktype': 'title', 'page': [3], 'tag': '',...",21,23,100044,1,23,[3],[3],,title


In [5]:
emlap_id_filename_dict = dict(zip(emlap_metadata["no."], emlap_metadata["working_title"]))
emlap_id_filename_dict

{100001: 'Augurello, Chrysopoeia',
 100002: 'Pseudo-Lull, Secretis',
 100003: 'Pantheus, Ars Transmutatione',
 100004: 'Anon, Vera alchemiae',
 100005: 'Pantheus, Voarchadumia',
 100006: 'Savonarola, De arte conficiendi aquam vitae',
 100007: 'Anon, Rosarium philosophorum',
 100008: 'Severinus, Epistola',
 100009: 'Vegius, Inter inferiora corpora',
 100010: 'Bracesco, De alchimia dialogi duo',
 100011: 'Anon, De alchemia',
 100012: 'Gessner, Euonymus',
 100013: 'Ulstadt, Coelum',
 100014: 'Toxites, Spongia',
 100015: 'Gessner, Euonymus II',
 100016: 'Bonus, Pretiosa margarita novella',
 100017: 'Bodenstein, Isagoge',
 100018: 'Trevisan, Peri chemeias',
 100019: 'Ulstadt, De epidemia',
 100020: 'Dorn, Artificii chymistici',
 100021: 'Dorn, Clavis',
 100022: 'Anon, De alchimia opuscula ',
 100023: 'Paracelsus, Labyrinthus',
 100024: 'Fanianus, De arte metallicae',
 100025: 'Pseudo-Lull, Codicillus',
 100026: 'Garlandius, Compendium Alchemiae',
 100027: 'Paracelsus, Pyrophilia',
 100028: 

In [28]:
position_files_dict = {}
for id, fn in emlap_id_filename_dict.items():
    position_files_dict[id] = fn.replace(".pdf", "_recalculated.json")


In [6]:
textblocks_path = "../data/emlap_recalculated_sanitized_textblocks/"
os.listdir(textblocks_path)[:10]

['100025_Pseudo-Lull1563_Codicillus_MDZ_MBS_recalculated.json',
 '100061_Moffett1588_Nosomantica_Hippocrates_ONB_recalculated.json',
 '100011_Anon1541_De_alchemia_MDZ_MBS_recalculated.json',
 '100099_Francus1607_De_Arte_Chemica_VD17_Halle_recalculated.json',
 '100012_Gessner1552_Thesaurus_Euonymi_Philiatri_ER_ZZ_recalculated.json',
 '100075_Dorn1581_Fasciculus_Paracelsicae_MDZ_MBS_recalculated.json',
 '100051_Ruland1564_Medicina_practica_recens_MDZ_MDS_recalculated.json',
 '100002_Pseudo-Lull1518_De_secretis_naturae_MDZ_MBS_recalculated.json',
 '100057_Hagecius1585_De_cerevisia_GB_ONB_recalculated.json',
 '100060_Witestein1583_Disceptatio_philosophica_MDZ_MBS_recalculated.json']

In [8]:
textblocks_filenames = os.listdir(textblocks_path)

In [9]:
positions_data_dict = {}
for fn in textblocks_filenames:
    id = int(fn[:6])
    with open(textblocks_path + fn, "rb") as f:
        positions_data_dict[id] = json.load(f)

In [12]:
print(positions_data_dict[100044][3][4])

{'coordinates': {'upper_left_x': 0.24484032382484244, 'upper_left_y': 0.20411399424500817, 'lower_right_x': 0.6592873453092174, 'lower_right_y': 0.20995722247415938}, 'text': 'rum omnium, in uniuersum ', 'tag': 'text'}


In [15]:
import numpy as np
import pandas as pd

def _flat_int_list(x):
    # turn scalars/arrays/Series/[arrays] into a flat list of ints
    if isinstance(x, (list, tuple)):
        seq = x
    elif isinstance(x, (np.ndarray, pd.Series)):
        seq = x.tolist()
    else:
        seq = [x]
    out = []
    for v in seq:
        if isinstance(v, (list, tuple, np.ndarray, pd.Series)):
            out.extend([int(i) for i in np.array(v).flatten().tolist()])
        else:
            out.append(int(v))
    return out

def get_positions(row):
    data = positions_data_dict[int(row["id"])]          # e.g. positions_data_dict[100044]
    pages  = _flat_int_list(row["page"])
    blocks = _flat_int_list(row["textblock"])

    # one page, many blocks → repeat page
    if len(pages) == 1 and len(blocks) > 1:
        pages = pages * len(blocks)

    out = []
    for p, b in zip(pages, blocks):
        page_blocks = data[p]                      # list of blocks on this page
        # simple 1-based fallback if needed
        if b >= len(page_blocks) and b > 0:
            b = b - 1
        # final guard: skip if still OOB
        if 0 <= b < len(page_blocks):
            out_el = page_blocks[b]
            out_el["page"] = p
            out.append(out_el)
        # else: silently skip (or append None if you prefer)
    return out

# apply
emlap_tokens_df["positions"] = emlap_tokens_df.apply(get_positions, axis=1)

In [16]:
emlap_tokens_df.sample(100).to_csv("../data/positions_full_sample.csv")

In [20]:
emlap_tokens_df.to_parquet("../data/emlap_corpus_public/emlap_tokens_positions_df.parquet")

In [18]:
import os

path = "../data/emlap_corpus_public/emlap_tokens_positions_df.parquet"
size_bytes = os.path.getsize(path)
size_mb = size_bytes / (1024 ** 2)

print(f"{size_mb:.2f} MB")

148.09 MB


In [30]:
# a sanity check
emlap_tokens_df[emlap_tokens_df["lemma"].str.contains("alchy")].head(30)

,id,token,lemma,pos,char_position,sent_len,pdf_page,pdf_page_textblock
76404,100034,Alchymia,alchymia,PROPN,"[52, 60]",67,[77],[22]
81331,100034,alchymistarum,alchymista,DET,"[36, 49]",58,[106],"[19, 20]"
137230,100060,alchymica,alchymicus,ADJ,"[131, 140]",252,[150],[23]
142127,100060,Alchymia,alchymia,PROPN,"[213, 221]",1006,[177],[9]
146721,100060,Alchymiam,alchymia,PROPN,"[595, 604]",834,[200],[30]
148404,100060,Alchymia,alchymia,PROPN,"[48, 56]",1369,[209],[6]
148430,100060,Alchymia,alchymia,PROPN,"[229, 237]",1369,[209],[11]
149967,100060,Alchymiam,alchymia,PROPN,"[145, 154]",1212,[216],[31]
150353,100060,Alchymiam,alchymia,PROPN,"[101, 110]",528,[218],[26]
150373,100060,Alchymia,alchymia,PROPN,"[216, 224]",528,[219],[1]


In [33]:
emlap_lemma_counts = pd.DataFrame(emlap_tokens_df[emlap_tokens_df["pos"].isin(["NOUN", "VERB", "ADJ", "PROPN"])].value_counts("lemma"))

In [34]:
len(emlap_lemma_counts)

66927

In [35]:
emlap_lemma_counts.head(5)

,count
lemma,
,22091
aqua,19987
facio,16031
possum,14364
dico,14091


In [36]:
emlap_lemma_counts.reset_index(inplace=True)

In [37]:
emlap_lemma_counts[emlap_lemma_counts["lemma"].str.startswith("alch")]

,lemma,count
811,alchimia,319
2324,alchymia,91
2789,alchimista,72
3319,alchymista,56
3644,alchemia,50
3914,alchimicus,44
4497,alchymicus,36
4725,alchimius,34
8508,alchemista,13
10659,alchimisticus,9
